# Detailed EDA — Receiver General Accounting Transactions

## Objective

Explore the statistical, categorical, temporal, and financial behavior of the Receiver General dataset after the [completed inventory](01_dataset_inventory.ipynb). This notebook develops the inventory findings without replacing or reproducing the entire inventory.

Raw data remains unchanged. All dates, grouping labels, and descriptive candidate flags are temporary analytical variables in separate objects; original columns are never overwritten. Extreme values are investigation candidates, not automatically errors, fraud, or accounting anomalies. Debit/credit codes are accounting directions, not automatically expenses/revenues or cash inflows/outflows.

**Interpretation constraints:** exact fiscal-calendar alignment, relationships between summary and item-level record types, and the complete Kaggle/local-extract provenance chain remain unresolved. The source structure accommodates voucher entries and accounting/control summaries; it is not uniformly one underlying transaction per row. Accounting Effective Date is the chronological variable; Fiscal Month is not converted into a calendar month.

Throughout, “records” means CSV rows. Amount totals are sums of recorded unsigned magnitudes, **not** reconciled revenue, expenditure, cash flow, or unique underlying economic value. Potential overlap among summary and item records is unresolved. Currency/units are not asserted. Concentration describes this extract, not risk. No cleaning, imputation of source values, model fitting, or application implementation is performed.

Documentation already reviewed in the inventory: [official catalogue](https://open.canada.ca/data/en/dataset/fa79f13c-2796-41a9-a9cd-28650ebe0f06), [field dictionary](https://donnees-data.tpsgc-pwgsc.gc.ca/ba1/SGBT-BCMS-dd.xml), and [supporting guide](https://donnees-data.tpsgc-pwgsc.gc.ca/ba1/tcrg-rgat/soutien-support-eng.html). This EDA does not reinterpret the unresolved documentation through inferred mappings.

## 1. Setup and Data Loading

Run from the notebook directory (or project root). Use the project's existing environment. The plotting dependency is checked explicitly. A deep snapshot plus file hashes support the preservation checks at the end. Short column aliases exist only in the separate `eda` DataFrame. Numeric identifiers are categorical grouping variables, never continuous measures.

In [ ]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

project_root = Path.cwd() if (Path.cwd() / "data/raw").exists() else Path.cwd().parent
raw_path = project_root / "data/raw/Receiver General Accounting Transactions/tcrg-rgat.csv"
inventory_path = project_root / "notebooks/01_dataset_inventory.ipynb"
protected_paths = [inventory_path] + sorted(p for p in (project_root / "data/raw").rglob("*") if p.is_file())
initial_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
raw_df = pd.read_csv(raw_path)
raw_snapshot = raw_df.copy(deep=True)
expected_columns = [
    "Accounting-Control-Number-Numéro-contrôle-comptable",
    "Journal-Voucher-Type-Code-Code-du-type-de-pièce-de-journal",
    "Fiscal-Year-Année-Fiscale",
    "Journal-Voucher-Identifier-Identificateur-de-la-pièce-de-journal",
    "Accounting-Effective-Date-Date-d'entrée-en-vigueur-comptable",
    "DepartmentNumber-Numéro-de-Ministère",
    "Fiscal-Month-Mois-Fiscal",
    "Journal-Voucher-Item-Identifier-Identificateur-de-l'item-de-la-pièce-de-journal",
    "Financial-Coding-Block-Bloc-de-codage-financier",
    "Subleger-Account-Identifier-Compte-du-grand-livre-auxiliaire",
    "Credit/Debit-Code-Code-Crédit/Débit",
    "Journal-Voucher-Item-Amount-Montant-de-l'item-de-la-pièce-de-journal",
    "Control-Data-Type-Type-de-données-de-contrôle",
    "Accounting-Summary-Transaction-Line-Count-Sommaire-comptable-transaction-compte-ligne",
    "GBS-Control-Number-Numéro-de-contrôle-du-SBG",
    "General-Ledger-Account-Code-Code-du-compte-du-grand-livre-général",
]
assert raw_df.columns.tolist() == expected_columns
assert raw_df.shape == (9282, 16)
aliases = ["control", "voucher_type", "fiscal_year", "voucher_id", "effective_date_text",
           "department", "fiscal_month", "item", "coding_block", "subledger", "direction",
           "amount", "control_type", "summary_lines", "gbs", "ledger"]
eda = raw_df.rename(columns=dict(zip(expected_columns, aliases))).copy(deep=True)
effective_dates = pd.to_datetime(eda["effective_date_text"], format="%Y-%m-%d", errors="coerce")
eda = eda.assign(effective_date=effective_dates, calendar_month=effective_dates.dt.to_period("M").astype(str))
amount = eda["amount"]
plt.rcParams.update({"figure.figsize": (10, 4), "axes.titlesize": 12, "axes.labelsize": 10})
print("Shape:", raw_df.shape)
print("Environment:", sys.version.split()[0], "pandas", pd.__version__, "matplotlib", matplotlib.__version__)
print("Raw SHA-256:", initial_hashes[str(raw_path)])
print("Exact expected headers confirmed:", len(expected_columns))

## 2. EDA Baseline Validation

Reproduce only the core inventory checks. The expected missing counts are taken from the existing inventory, including fields with zero missing values. Parsing failures are counted, never repaired.

In [ ]:
expected_missing = pd.Series([0, 26, 0, 9104, 0, 26, 0, 0, 9000, 2186, 0, 0, 178, 26, 239, 9091], index=expected_columns)
pd.testing.assert_series_equal(raw_df.isna().sum(), expected_missing)
baseline = pd.Series({
    "rows": len(raw_df), "columns": raw_df.shape[1],
    "complete_duplicate_occurrences": int(raw_df.duplicated().sum()),
    "missing_cells": int(raw_df.isna().sum().sum()),
    "date_parse_failures": int(effective_dates.isna().sum()),
    "candidate_key_duplicate_occurrences": int(eda.duplicated(["control", "item"]).sum()),
    "candidate_key_rows_with_missing_components": int(eda[["control", "item"]].isna().any(axis=1).sum()),
})
assert baseline.to_dict() == {"rows": 9282, "columns": 16, "complete_duplicate_occurrences": 0,
    "missing_cells": 29876, "date_parse_failures": 0, "candidate_key_duplicate_occurrences": 0,
    "candidate_key_rows_with_missing_components": 0}
assert effective_dates.min() == pd.Timestamp("2023-04-03")
assert effective_dates.max() == pd.Timestamp("2024-01-03")
display(baseline.to_frame("observed"))
print("Baseline matches the inventory. Uniqueness applies to this extract only.")

## 3. Financial Amount Distribution

Quantiles use Pandas' default linear interpolation. Skewness is descriptive sample skewness, not a formal normality test. The first histogram displays only values at or below the 95th percentile; the second includes every positive amount on a log10 display. Zero amounts are reported separately. Neither view removes or changes stored values.

In [ ]:
amount_stats = amount.describe(percentiles=[.01, .05, .25, .5, .75, .9, .95, .99, .995, .999])
display(amount_stats.to_frame("recorded_amount"))
print("Sample skewness:", amount.skew())
print("Zero / negative / non-finite counts:", amount.eq(0).sum(), amount.lt(0).sum(), (~np.isfinite(amount)).sum())
q95, q99 = amount.quantile(.95), amount.quantile(.99)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(amount[amount <= q95], bins=45, color="#3274a1")
axes[0].set(title="Amount distribution: view through 95th percentile", xlabel="Recorded amount (unconfirmed units)", ylabel="Record count")
axes[1].hist(np.log10(amount[amount > 0]), bins=50, color="#287d6c")
axes[1].set(title="Positive amounts: complete log10 display", xlabel="log10(recorded amount)", ylabel="Record count")
fig.tight_layout(); plt.show(); plt.close(fig)
display(Markdown(f"**Observation:** median {amount.median():,.2f}, mean {amount.mean():,.2f}, maximum {amount.max():,.2f}, skewness {amount.skew():.2f}. The upper tail strongly dominates the mean. The percentile view omits {(amount > q95).sum():,} records only from that panel; the log panel omits {amount.eq(0).sum()} zeros only from logarithmic display. All records remain available."))

## 4. Debit / Credit Analysis

All grouped sums retain recorded signs (here amounts are nonnegative); no DR/CR sign transformation or netting is applied. Missing category labels are retained in grouped tables through `dropna=False`. The shared summary includes quartiles and skewness.

In [ ]:
def summarize_amount(by):
    grouped = eda.groupby(by, dropna=False, observed=True)["amount"]
    result = grouped.agg(records="size", total="sum", mean="mean", median="median", std="std", minimum="min", maximum="max", skewness="skew")
    result["q25"] = grouped.quantile(.25)
    result["q75"] = grouped.quantile(.75)
    return result

direction_stats = summarize_amount("direction")
display(direction_stats)
fig, ax = plt.subplots(figsize=(7, 4))
labels = direction_stats.index.tolist()
ax.boxplot([np.log10(eda.loc[(eda.direction == label) & (amount > 0), "amount"]) for label in labels], tick_labels=labels, showfliers=True)
ax.set(title="Debit/credit amount shapes: positive records", xlabel="Accounting direction", ylabel="log10(recorded amount)")
fig.tight_layout(); plt.show(); plt.close(fig)
display(Markdown(f"**Observation:** DR has {direction_stats.loc['DR', 'records']:,.0f} records and {direction_stats.loc['DR', 'total']/amount.sum()*100:.2f}% of recorded magnitude, but DR/CR medians are {direction_stats.loc['DR', 'median']:,.2f} / {direction_stats.loc['CR', 'median']:,.2f}. Similar centers coexist with very different upper tails. **Implication:** direction is useful for stratification, but not sufficient by itself. These are descriptively different observed distributions; independence and formal population differences have not been established. DR is not labeled expense and CR is not labeled revenue."))

## 5. Journal Voucher Type Analysis

Codes remain labels without inferred business meanings. “Rare” below means at most 10 records in this extract, an exploratory reporting threshold only. All categories, including missing labels, appear in the table; plotted categories are selected by stated rankings.

In [ ]:
type_stats = summarize_amount("voucher_type")
display(type_stats.sort_values("records", ascending=False))
rare_types = type_stats.loc[type_stats.records <= 10]
print("Observed non-null voucher types:", eda.voucher_type.nunique(), "Missing labels:", eda.voucher_type.isna().sum())
print("Rare groups (including missing if eligible):", len(rare_types))
display(rare_types[["records", "median", "maximum"]])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
type_stats.nlargest(8, "records")["records"].sort_values().plot.barh(ax=axes[0], color="#3274a1")
axes[0].set(title="Eight most frequent voucher types", xlabel="Record count", ylabel="Voucher code")
type_stats.nlargest(8, "total")["total"].sort_values().plot.barh(ax=axes[1], color="#287d6c")
axes[1].set(title="Eight largest voucher-type magnitude totals", xlabel="Sum of recorded amounts (unconfirmed units)", ylabel="Voucher code")
fig.tight_layout(); plt.show(); plt.close(fig)
display(Markdown(f"**Observation:** ADEPT leads counts ({type_stats.loc['ADEPT','records']:,.0f}) with median {type_stats.loc['ADEPT','median']:,.2f}; JVBDP leads recorded magnitude with only {type_stats.loc['JVBDP','records']:,.0f} rows and median {type_stats.loc['JVBDP','median']:,.2f}. Voucher type separates amount scales much more strongly than the overall DR/CR medians. Rare codes do not become errors merely because they are rare."))

## 6. Department Analysis

Department numbers are identifiers, not measurements. No department identity is inferred. Counts refer to records, not independently verified business transactions. Missing department labels remain a separate group.

In [ ]:
department_stats = summarize_amount("department")
display(department_stats.sort_values("records", ascending=False))
known_department = department_stats.loc[department_stats.index.notna()]
print("Known departments:", len(known_department))
print("Missing department records:", eda.department.isna().sum())
top5_department_record_share = known_department.records.nlargest(5).sum()/len(eda)*100
top2_department_amount_share = known_department.total.nlargest(2).sum()/amount.sum()*100
print("Top 5 known departments, share of all records:", top5_department_record_share)
print("Top 2 known departments, share of all recorded magnitude:", top2_department_amount_share)
display(Markdown(f"**Observation:** the five busiest known departments contain {top5_department_record_share:.2f}% of rows; departments 97 and 6 together account for {top2_department_amount_share:.2f}% of recorded magnitude. Department 122 leads record count, while 97 leads magnitude. Activity and amount concentration measure different things. Mixed voucher types and summary granularity may explain part of the difference; no risk conclusion follows."))

## 7. Accounting Dimensions

Coverage is reported before any account analysis. Tables describe only the nonmissing subset for each dimension, with its size shown. Absence is not imputed. Source documentation permits some record-type-dependent blanks, but this does not establish the reason for every missing local value.

In [ ]:
account_fields = ["ledger", "subledger", "coding_block", "control_type"]
coverage = pd.DataFrame({"populated_records": eda[account_fields].notna().sum(), "coverage_percent": eda[account_fields].notna().mean()*100, "known_categories": eda[account_fields].nunique()})
display(coverage)
for field in account_fields:
    table = summarize_amount(field)
    known = table.loc[table.index.notna()]
    print(field, "— top 8 known categories by record count; full known-category count:", len(known))
    display(known.nlargest(8, "records")[["records", "total", "median", "maximum"]])
print("Summary-line count description (known values only):")
display(eda.summary_lines.describe().to_frame("summary_lines"))
print("Known counts >1 / zero / missing:", eda.summary_lines.gt(1).sum(), eda.summary_lines.eq(0).sum(), eda.summary_lines.isna().sum())

**Interpretation:** Control Data Type has broad coverage and may help distinguish accounting processes. Subledger offers a larger set of potential account peers but its missingness must be examined by record type and amount, not just row percentage. General Ledger Account and Financial Coding Block describe small selected subsets; their apparent distributions cannot represent the entire extract. Summary-line counts provide context about grouped detail, but dividing amounts by counts would not establish comparable individual transactions. No such reconstruction is performed.

## 8. Temporal Analysis

Calendar periods derive exclusively from Accounting Effective Date. Daily tables contain observed dates only; dates with no rows are reported as absent from the extract, not imputed as zero financial activity. January 2024 is an endpoint fragment, not a full comparable month. Fiscal labels are untouched and their exact alignment remains unresolved.

In [ ]:
daily = summarize_amount("effective_date").sort_index()
monthly = summarize_amount("calendar_month").sort_index()
display(monthly[["records", "total", "median", "q25", "q75", "maximum"]])
calendar_days = pd.date_range(effective_dates.min(), effective_dates.max(), freq="D")
absent_dates = calendar_days.difference(pd.DatetimeIndex(daily.index))
print("Observed dates:", len(daily), "Calendar days in observed interval:", len(calendar_days), "Dates with no rows:", len(absent_dates))
print("Days represented by month:")
display(eda.groupby("calendar_month").effective_date.agg(["min", "max", "nunique"]))
print("Five largest daily magnitude totals:"); display(daily.nlargest(5,"total")[["records","total","median","maximum"]])
print("Five busiest dates:"); display(daily.nlargest(5,"records")[["records","total","median"]])
print("Five quietest observed dates:"); display(daily.nsmallest(5,"records")[["records","total","median"]])
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, field, title, ylabel in zip(axes, ["records", "total", "median"], ["Calendar-month record counts", "Calendar-month recorded magnitude totals", "Calendar-month median amounts"], ["Records", "Sum of amounts", "Median amount"]):
    ax.plot(monthly.index, monthly[field], marker="o")
    ax.set(title=title, ylabel=ylabel)
axes[-1].set_xlabel("Accounting Effective Date calendar month; endpoints may be partial")
axes[-1].tick_params(axis="x", rotation=45)
fig.tight_layout(); plt.show(); plt.close(fig)
display(Markdown(f"**Observation:** {monthly.records.idxmax()} has the most rows ({monthly.records.max():,.0f}); {monthly.total.idxmax()} has the largest monthly recorded total. January 2024 has only {monthly.loc['2024-01','records']:,.0f} records through January 3. The highest daily total occurs on {daily.total.idxmax():%Y-%m-%d}. Quiet dates, gaps, and spikes are descriptive; completeness, holidays, reporting schedules, and economic causes are unverified. No annual seasonality is established by less than one year of effective dates."))

## 9. Temporal × Financial Relationships

Use a fixed extract-wide 99th percentile to describe high-value activity consistently across months. This is a descriptive cutoff, not a learned anomaly threshold. Report counts and rates because monthly record counts differ. Type and department mix help contextualize changes in period totals.

In [ ]:
eda_high = eda.assign(above_global_p99=amount > q99)
high_by_month = eda_high.groupby("calendar_month").above_global_p99.agg(["size", "sum", "mean"])
high_by_month.columns = ["records", "above_p99_records", "above_p99_fraction"]
display(high_by_month)
month_direction = eda.groupby(["calendar_month","direction"]).amount.agg(["size","sum","median"])
display(month_direction)
top_types = type_stats.nlargest(5,"records").index
print("Calendar-month counts for five most frequent types:")
display(pd.crosstab(eda.calendar_month, eda.voucher_type).reindex(columns=top_types))
print("Calendar-month counts for five busiest known departments:")
display(pd.crosstab(eda.calendar_month, eda.department).reindex(columns=known_department.nlargest(5,"records").index))
fig, axes = plt.subplots(1,2,figsize=(12,4))
month_direction["median"].unstack().plot(ax=axes[0],marker="o")
axes[0].set(title="Direction-specific monthly median", xlabel="Calendar month", ylabel="Median recorded amount")
high_by_month["above_p99_fraction"].mul(100).plot.bar(ax=axes[1],color="#a65c35")
axes[1].set(title="Monthly share above global 99th percentile", xlabel="Calendar month", ylabel="Percent of month's records")
fig.tight_layout(); plt.show(); plt.close(fig)
display(Markdown(f"**Observation:** the fixed cutoff is {q99:,.2f}. Above-cutoff counts range from {high_by_month.above_p99_records.min()} to {high_by_month.above_p99_records.max()} per observed month. Changes in direction/type/department composition can affect aggregate behavior. **Hypothesis:** some temporal variation is compositional; the tables do not establish causation or an adjusted trend."))

## 10. Concentration Analysis

Rank known categories separately from missing labels. Shares of all records/amounts use the whole extract as denominator; within-known shares use only populated labels. Missing accounts are never treated as a single real account. Cumulative shares are sums of recorded magnitudes and may include unresolved overlapping summaries.

In [ ]:
concentration_rows = []
concentration_tables = {}
for field in ["department","voucher_type","subledger","control_type","control"]:
    table = summarize_amount(field)
    known = table.loc[table.index.notna()].sort_values("total",ascending=False).copy()
    known["percent_all_amount"] = known.total / amount.sum()*100
    known["cumulative_percent_all_amount"] = known.percent_all_amount.cumsum()
    known["cumulative_percent_known_amount"] = known.total.cumsum()/known.total.sum()*100
    concentration_tables[field] = known
    missing = eda[field].isna()
    concentration_rows.append({"dimension":field,"known_groups":len(known),
        "missing_records_percent":missing.mean()*100,"missing_amount_percent":amount[missing].sum()/amount.sum()*100,
        "top1_known_amount_percent_all":known.total.nlargest(1).sum()/amount.sum()*100,
        "top5_known_amount_percent_all":known.total.nlargest(5).sum()/amount.sum()*100,
        "top5_known_records_percent_all":known.records.nlargest(5).sum()/len(eda)*100})
    print(field, "— top five known groups ranked by recorded amount")
    display(known[["records","total","percent_all_amount","cumulative_percent_all_amount","cumulative_percent_known_amount"]].head(5))
concentration = pd.DataFrame(concentration_rows).set_index("dimension")
display(concentration)
fig, ax = plt.subplots(figsize=(8,4))
for field in ["department","voucher_type","subledger"]:
    values = concentration_tables[field].cumulative_percent_all_amount
    ax.plot(range(1,min(10,len(values))+1),values.iloc[:10],marker="o",label=field)
ax.set(title="Known-category cumulative contribution to all recorded magnitude", xlabel="Number of largest known categories", ylabel="Cumulative percent of all recorded amounts", ylim=(0,105))
ax.legend(); fig.tight_layout(); plt.show(); plt.close(fig)
display(Markdown(f"**Observation:** the five largest known voucher types contribute {concentration.loc['voucher_type','top5_known_amount_percent_all']:.2f}% of magnitude. Missing subledger labels account for {concentration.loc['subledger','missing_records_percent']:.2f}% of records but {concentration.loc['subledger','missing_amount_percent']:.2f}% of magnitude. Apparent account concentration is therefore highly coverage-dependent. Accounting-control grouping is descriptive only; it does not deduplicate underlying activity."))

## 11. Missingness Patterns

Missingness is measured on original values and summarized by voucher type, direction, department, calendar month, and control type. Tables show group size alongside missing percentages. Associations are observed patterns; neither random missingness nor a definitive structural cause is established by these summaries.

In [ ]:
missing_fields = ["voucher_id","ledger","coding_block","subledger","control_type","summary_lines","department","gbs"]
missing_flags = eda[missing_fields].isna()
missing_by = {}
for dimension in ["voucher_type","direction","department","calendar_month","control_type"]:
    grouped_flags = missing_flags.groupby(eda[dimension],dropna=False,observed=True)
    rates = grouped_flags.mean()*100
    rates.insert(0,"records",grouped_flags.size())
    missing_by[dimension] = rates
    print(dimension, "— missing percentages; department/control-type display limited to 12 largest groups")
    display(rates.sort_values("records",ascending=False).head(12) if dimension in ["department","control_type"] else rates)
print("Most common joint missingness patterns (True = missing):")
display(missing_flags.value_counts().head(10).rename("records").to_frame())
print("Voucher-reference presence versus control-type presence:")
display(pd.crosstab(eda.voucher_id.notna().rename("voucher_id_present"), eda.control_type.notna().rename("control_type_present")))
fig, ax = plt.subplots(figsize=(10,6))
heat = missing_by["voucher_type"].nlargest(12,"records")[missing_fields]
image = ax.imshow(heat.to_numpy(),vmin=0,vmax=100,cmap="Blues",aspect="auto")
ax.set_yticks(range(len(heat)),labels=[str(x) for x in heat.index])
ax.set_xticks(range(len(missing_fields)),labels=missing_fields,rotation=45,ha="right")
ax.set(title="Missingness in the 12 most frequent voucher-type groups", xlabel="Field (temporary alias)", ylabel="Voucher-type group")
fig.colorbar(image,ax=ax,label="Missing percent within group")
fig.tight_layout(); plt.show(); plt.close(fig)

**Observation:** the tables show category-dependent coverage, including voucher types with entirely absent subledger, ledger, or coding fields. The joint patterns and voucher-reference/control-type cross-tab distinguish field-presence configurations without assigning record-type labels.

**Documented expectation:** the inventory's dictionary review permits summary-line blanks for voucher transactions and distinguishes control data from voucher entries. **Hypothesis:** some observed missingness reflects these different processes. The exact local row taxonomy and the cause of every blank remain unknown. It would be inappropriate to apply a blanket fill rule or declare missingness random from the overall percentages.

## 12. Extreme Value Investigation

Investigate both ends, retaining original row indices and accounting context. Global p99 and Tukey upper fences (`Q3 + 1.5 × IQR`) are descriptive screening rules, not error labels. Peer fences use direction × voucher type only where there are at least 30 records and a positive IQR; 30 is an exploratory stability guard, not proof of statistical adequacy. No lower-tail rule is applied where a negative fence would be uninformative for nonnegative amounts.

In [ ]:
q1, q3 = amount.quantile(.25), amount.quantile(.75)
global_upper_fence = q3 + 1.5*(q3-q1)
peer_keys = ["direction","voucher_type"]
peer_grouped = eda.groupby(peer_keys,dropna=True).amount
peer_n = peer_grouped.transform("size")
peer_q1 = peer_grouped.transform(lambda s:s.quantile(.25))
peer_q3 = peer_grouped.transform(lambda s:s.quantile(.75))
peer_eligible = peer_n.ge(30) & peer_q3.gt(peer_q1)
peer_upper_fence = peer_q3 + 1.5*(peer_q3-peer_q1)
global_candidates = amount > global_upper_fence
peer_candidates = peer_eligible & amount.gt(peer_upper_fence)
extreme_summary = pd.Series({"global_IQR_upper_fence":global_upper_fence,
    "above_global_IQR_count":int(global_candidates.sum()),"global_p99":q99,
    "above_global_p99_count":int((amount>q99).sum()),"peer_eligible_records":int(peer_eligible.sum()),
    "above_peer_IQR_count":int(peer_candidates.sum()),
    "global_IQR_only_among_eligible":int((peer_eligible & global_candidates & ~peer_candidates).sum()),
    "peer_IQR_only_among_eligible":int((peer_candidates & ~global_candidates).sum())})
display(extreme_summary.to_frame("value"))
print("Global/peer comparison only within eligible peer populations:")
display(pd.crosstab(global_candidates[peer_eligible].rename("above_global_fence"),peer_candidates[peer_eligible].rename("above_peer_fence")))
context_fields = ["control","item","effective_date","direction","voucher_type","department","amount","subledger","ledger","coding_block","control_type","summary_lines","voucher_id"]
print("Ten largest records; original zero-based row index retained:")
display(eda.nlargest(10,"amount")[context_fields])
print("Ten smallest records including zeros:")
display(eda.nsmallest(10,"amount")[context_fields])
print("Peer-only candidates: five largest examples:")
display(eda.loc[peer_candidates & ~global_candidates].nlargest(5,"amount")[context_fields])
print("Global-only candidates within eligible peers: five largest examples:")
display(eda.loc[peer_eligible & global_candidates & ~peer_candidates].nlargest(5,"amount")[context_fields])
display(Markdown(f"**Observation:** the global IQR rule selects {global_candidates.sum():,} records ({global_candidates.mean()*100:.2f}%), versus {peer_candidates.sum():,} peer-rule candidates among {peer_eligible.sum():,} eligible records. These denominators differ. There are {amount.eq(0).sum()} zeros; the maximum record belongs to {eda.loc[amount.idxmax(),'voucher_type']} and has summary-line count {eda.loc[amount.idxmax(),'summary_lines']:g}. **Implication:** mixture and aggregation context make a universal cutoff questionable. Neither rule establishes anomalies; peer rules can also fail with skew, small samples, dependence, or mixed record types."))

## 13. Potential Anomaly-Detection Groups

Compare group coverage and size before recommending future peers. Only rows with all selected grouping fields populated enter a given group-size calculation; excluded rows remain in the dataset and are counted. Thresholds here describe feasibility, not an approved model design.

In [ ]:
peer_options = [["direction"],["voucher_type"],["direction","voucher_type"],
    ["direction","voucher_type","department"],["direction","control_type"],
    ["direction","subledger"],["direction","ledger"],["direction","voucher_type","calendar_month"]]
peer_feasibility_rows = []
for keys in peer_options:
    complete = eda[keys].notna().all(axis=1)
    sizes = eda.loc[complete].groupby(keys,observed=True).size()
    peer_feasibility_rows.append({"grouping":" × ".join(keys),"covered_records":int(complete.sum()),
        "excluded_missing_records":int((~complete).sum()),"groups":len(sizes),"singleton_groups":int(sizes.eq(1).sum()),
        "median_group_size":sizes.median(),"groups_ge30":int(sizes.ge(30).sum()),
        "records_in_groups_ge30":int(sizes[sizes.ge(30)].sum())})
peer_feasibility = pd.DataFrame(peer_feasibility_rows).set_index("grouping")
display(peer_feasibility)

**Candidate hierarchy:** start by examining direction × voucher type, then department where enough comparable records remain. Direction × control type is a useful alternative whose redundancy with voucher type is checked below. Subledger may support selected covered subsets, but does not cover much of the recorded magnitude. Ledger/coding peers have particularly selective coverage. Calendar period is useful context; crossing every group with month can fragment already small groups.

These are future investigation populations, not validated anomaly models. Record-type/summary relationships must be understood before treating observations as independent peers. Missing labels need their own review, not invented categories that pretend to be real accounts. No ML model is built or trained.

## 14. Relationship Analysis

Evaluate associations with plausible meaning: voucher type versus control type, type/department composition, and amount versus documented grouped-detail volume. Do not correlate amounts against numeric identifiers. Spearman correlation below uses two quantitative fields only, excludes missing pairs explicitly, and is descriptive rather than causal.

In [ ]:
type_control_counts = eda.loc[eda.voucher_type.notna() & eda.control_type.notna()].groupby(["voucher_type","control_type"]).size()
type_control_cardinality = type_control_counts.groupby(level=0).size().rename("distinct_control_types")
print("Voucher types with more than one observed control type:")
display(type_control_cardinality[type_control_cardinality>1].to_frame())
print("Rows with both codes:", int((eda.voucher_type.notna() & eda.control_type.notna()).sum()))
print("Direction/type/department groups with greatest magnitude:")
display(summarize_amount(["direction","voucher_type","department"]).nlargest(10,"total")[["records","total","median","maximum"]])
line_complete = eda[["amount","summary_lines"]].notna().all(axis=1)
line_spearman = eda.loc[line_complete,["amount","summary_lines"]].corr(method="spearman").loc["amount","summary_lines"]
print("Amount vs summary-line count Spearman; pairs:", int(line_complete.sum()), "rho:", line_spearman)
line_groups = eda.assign(line_count_context=pd.cut(eda.summary_lines,bins=[-np.inf,0,1,10,np.inf],labels=["nonpositive","1","2–10",">10"]))
display(line_groups.groupby("line_count_context",observed=True,dropna=False).amount.agg(["size","median","mean","max"]))
print("Within-type Spearman where n>=30 and summary lines vary:")
line_relations=[]
for label, group in eda.loc[line_complete & eda.voucher_type.notna()].groupby("voucher_type"):
    if len(group)>=30 and group.summary_lines.nunique()>1:
        line_relations.append({"voucher_type":label,"records":len(group),"rho":group[["amount","summary_lines"]].corr(method="spearman").loc["amount","summary_lines"]})
display(pd.DataFrame(line_relations))

**Interpretation:** shared type/control classifications may offer redundant grouping information; exceptions should be retained for review rather than forced into a one-to-one mapping. Differences between pooled and within-type amount/line-count associations can reflect category mixture and aggregation. A larger grouped amount may reflect more underlying detail, but line counts do not reveal the distribution of that detail. No division, reconstruction, or deduplication of underlying transactions is justified here.

## 15. Key EDA Findings

This table is generated from the preceding calculations so its numbers remain traceable to executed outputs.

In [ ]:
findings = pd.DataFrame([
    ["Strong amount skew", f"Median {amount.median():,.2f}; mean {amount.mean():,.2f}; skewness {amount.skew():.2f}", "Use distribution-aware summaries and contextual comparisons"],
    ["Direction tails differ", f"DR contributes {direction_stats.loc['DR','total']/amount.sum()*100:.2f}% of recorded magnitude", "Direction is useful context, not expense/revenue classification"],
    ["Voucher types separate scales", f"ADEPT median {type_stats.loc['ADEPT','median']:,.2f}; JVBDP median {type_stats.loc['JVBDP','median']:,.2f}", "Investigate voucher-type peers before global thresholds"],
    ["Department concentration", f"Top 2 known departments: {top2_department_amount_share:.2f}% of magnitude; top 5 by count: {top5_department_record_share:.2f}% of rows", "Distinguish activity concentration from value concentration"],
    ["Subledger selection limits", f"Missing labels carry {concentration.loc['subledger','missing_amount_percent']:.2f}% of magnitude", "Account peers represent a selected subset"],
    ["Short and partial temporal coverage", f"{len(daily)} observed dates; January 2024 has {monthly.loc['2024-01','records']:,.0f} rows through January 3", "No full-year seasonality or complete-month January comparison"],
    ["Context-sensitive extreme candidates", f"Global IQR selects {global_candidates.sum():,}; peer IQR selects {peer_candidates.sum():,} within {peer_eligible.sum():,} eligible rows", "Investigate rule disagreement, not automatic removal"],
    ["Missingness varies by category", "Group missingness tables and joint presence patterns in section 11", "Validate structural explanations before cleaning rules"],
    ["Inventory baseline reproduced", "9,282 × 16; zero full duplicates; composite key unique; all dates parse", "No baseline correction is needed"],
],columns=["Finding","Evidence","Potential Relevance"])
display(findings)

## 16. Data Quality Issues Identified for Later Investigation

| Status | EDA-supported issue or behavior | Later action to consider; not performed here |
|---|---|---|
| Confirmed limitation | Sparse ledger, voucher-reference, and coding fields; coverage is highly category-dependent | Document field applicability and inspect source rules before any imputation or dropping |
| Confirmed limitation | Subledger missingness covers a disproportionately large share of recorded magnitude | Assess representativeness of account-based analyses |
| Confirmed semantic-type concern | Department, subledger, voucher, and GBS identifiers load as numeric values, including floats | Verify source-preserving identifier representations and leading-zero requirements; no precision defect is inferred from dtype alone |
| Confirmed coverage limit | Effective dates span less than a year; January stops on January 3; dates with no rows exist | Establish extract boundaries and reporting completeness before gap treatment |
| Requires investigation | Zero and very small amounts, very large amounts, and global/peer threshold disagreements | Review accounting context and source evidence; do not remove values merely for extremeness |
| Requires investigation | Summary counts vary and pooled amounts mix accounting processes | Establish record-type relationships and possible overlap before reconciliation or economic totals |
| Requires investigation | Exact fiscal alignment and redistribution/extract provenance | Obtain source clarification; do not infer corrections from effective dates or control-number patterns |
| Expected/documented possibility, not verified row by row | Multiple rows can share an Accounting Control Number; some blanks depend on record type | Preserve legitimate grouping; the observed composite key remains a candidate only |

Baseline checks found no complete duplicates, no invalid effective-date parses, and no missing amount values. These checks do not establish complete business validity. Category codes and numeric extremes are not classified as errors. No cleaning decisions are enacted.

## 17. Implications for Financial Manager AI

| Capability | Inventory assessment retained | Effect of detailed EDA |
|---|---|---|
| Financial Data Analysis | Suitable | Strengthened for descriptive accounting exploration: category-specific amount behavior, concentration, temporal profiles, and coverage limits can be quantified. Recorded totals remain unreconciled magnitudes rather than business revenue/expense/cash facts. |
| Financial Anomaly Detection | Potentially Suitable | Stronger evidence for contextual peer investigation, but greater caution about global cutoffs, missingness, small groups, and unresolved summary/item relationships. No model performance, anomaly labels, or confirmed anomalous transactions are established. |

Other assessments remain unchanged: Accounts Payable **Inadequate**, Budget Variance **Inadequate**, Financial Document RAG **Inadequate**, Cash Flow Forecast **Partial**. No new obligations, budgets, document corpus, or confirmed cash-movement history were introduced by EDA. This notebook does not change the inventory or authorize operational implementation.

## 18. Preliminary EDA Conclusions

**Observations:** amounts are strongly right-skewed; similar DR/CR medians hide different tails. Voucher types and departments differ substantially in scale and frequency. Recorded magnitude is concentrated in a few categories, and missing subledger labels represent a large value share. Effective-date calendar periods show variable counts, totals, medians, and high-value frequencies, with a very short January endpoint. Missingness depends on observed categories. Descriptive global and peer rules select different candidate sets. Section 15 supplies the measured evidence.

**Hypotheses for later investigation:** part of the amount variation and missingness may reflect distinct accounting processes and grouped-detail volume. Period spikes may reflect category mix or reporting schedules. Direction × voucher type, refined by department where sample sizes permit, may yield more meaningful peers than global thresholds. Control type may add or duplicate context. These are hypotheses, not causal conclusions or a trained detection strategy.

**Unresolved questions:** exact fiscal-calendar alignment; precise relationships and overlap among summary and item records; complete Kaggle/local-extract provenance; currency/units and validated accounting semantics; extract completeness; and the availability of reference evidence for evaluating anomaly candidates. The source's mixed-record interpretation remains in force. No fiscal mapping or row-type classification is inferred into existence.

**Next decision, not an action:** review these findings before authorizing a cleaning plan, further source clarification, or model-related work. Do not automatically remove extremes, fill structural blanks, create database keys, or begin Company Financials EDA. The original capability assessments remain appropriate within their documented limits.

### End-to-End Preservation Checks

The final cell compares the original DataFrame against its deep snapshot and a fresh read, verifies every protected raw file and the inventory hash, and reconciles grouped counts/totals. It performs no data repair or export.

In [ ]:
pd.testing.assert_frame_equal(raw_df, raw_snapshot)
pd.testing.assert_frame_equal(raw_df, pd.read_csv(raw_path))
assert raw_df.columns.tolist() == expected_columns
for protected_path, original_hash in initial_hashes.items():
    assert hashlib.sha256(Path(protected_path).read_bytes()).hexdigest() == original_hash, protected_path
for table in [direction_stats,type_stats,department_stats,monthly,daily]:
    assert table.records.sum() == len(raw_df)
    assert np.isclose(table.total.sum(),amount.sum(),rtol=1e-12)
assert len(eda) == len(raw_df)
assert baseline["candidate_key_duplicate_occurrences"] == 0
assert not plt.get_fignums()
print("PASS: all raw files and inventory unchanged; original DataFrame values/types preserved; grouped totals reconcile.")
print("Temporary analytical variables remain in this notebook only. No cleaning, ML training, or application implementation performed.")